In [1]:
print("""
@File         : group_by_apply.ipynb
@Author(s)    : Stephen CUI
@LastEditor(s): Stephen CUI
@CreatedTime  : 2025-01-04 22:48:59
@Email        : cuixuanstephen@gmail.com
@Description  : group by apply
""")


@File         : group_by_apply.ipynb
@Author(s)    : Stephen CUI
@LastEditor(s): Stephen CUI
@CreatedTime  : 2025-01-04 22:48:59
@Email        : cuixuanstephen@gmail.com
@Description  : group by apply



In [2]:
import pandas as pd

An equivalent function for group by exists as `pd.core.groupby.DataFrameGroupBy.apply` with all of the same caveats. Generally, this function is overused, and you should opt for `pd.core.groupby.
DataFrameGroupBy.agg` or `pd.core.groupby.DataFrameGroupBy.transform` instead. However, for the cases where you don’t really want an aggregation or a transformation, but something in between,
using `apply` is your only option.

一般来说，`pd.core.groupby.DataFrameGroupBy.apply` 应该只作为最后的手段。它有时会产生模棱两可的行为，而且很容易在 pandas 的不同版本之间出现问题。

In [3]:
df = pd.DataFrame([
    ["group_a", 42],
    ["group_a", 555],
    ["group_a", 42],
    ["group_a", 555],
    ["group_b", 0],
], columns=["group", "value"])
df = df.convert_dtypes(dtype_backend="numpy_nullable")
df

,group,value
0,group_a,42
1,group_a,555
2,group_a,42
3,group_a,555
4,group_b,0


想要使用 agg 和 transform 实现以下效果是不可能的。原因很简单；聚合需要将每个组标签的值减少到单个值。在输出中重复两次标签 `group_a` 是聚合无法实现的。同样，transform 需要生成与调用 pd.DataFrame 共享相同行索引的结果，这也不是我们想要的。

In [4]:
pd.Series(
    [42, 555, 0],
    index=pd.Index(["group_a", "group_a", "group_b"], name="group"),
    dtype=pd.Int64Dtype(),
)

group
group_a     42
group_a    555
group_b      0
dtype: Int64

`pd.core.groupby.DataFrameGroupBy.apply` 是中间方法，可以让我们更接近期望的结果：

In [6]:
def mode_for_apply(df: pd.DataFrame):
    return df['value'].mode()

df.groupby('group').apply(mode_for_apply, include_groups=False)

group     
group_a  0     42
         1    555
group_b  0      0
Name: value, dtype: Int64

> 传递参数 `include_groups=False` 是为了抑制有关 pandas 2.2 中行为的任何弃用警告。在后续版本中，可能不需要这样做。

值得注意的是，我们将 `mode_for_apply` 函数的参数注释为 pd.DataFrame。使用聚合和转换时，用户定义函数一次只接收一个 pd.Series 数据，但使用 `apply` 时，会获得整个 pd.DataFrame。要更详细地了解发生了什么，可以向用户定义函数添加打印语句：

In [9]:
def mode_for_apply(df: pd.DataFrame):
    print(f'\nThe data passed to apply is:\n{type(df)}')
    return df['value'].mode()

In [10]:
df.groupby('group').apply(mode_for_apply, include_groups=False)


The data passed to apply is:
<class 'pandas.core.frame.DataFrame'>

The data passed to apply is:
<class 'pandas.core.frame.DataFrame'>


group     
group_a  0     42
         1    555
group_b  0      0
Name: value, dtype: Int64

Essentially, `pd.core.groupby.DataFrameGroupBy.apply` passes a pd.DataFrame of data to the user-defined function, excluding the column(s) that are used for grouping. From there, it will look at the return type of the user-defined function and try to infer the best possible output shape it can. In this particular instance, because our mode_for_apply function returns a pd.Series, `pd.core.groupby.DataFrameGroupBy.apply` has determined that the best output shape should have a `pd.MultiIndex`, where the first level of the index is the group value and the second level contains the row index from
the pd.Series returned by the `mode_for_apply` function.

`pd.core.groupby.DataFrameGroupBy.apply` 被过度使用的地方在于，当它检测到它所应用的函数归约到标量时，它可以改变其形状以看起来像聚合：

In [12]:
def sum_values(df: pd.DataFrame):
    return df['value'].sum()


df.groupby('group').apply(sum_values, include_groups=False)

group
group_a    1194
group_b       0
dtype: int64

然而，以这种方式使用它是个陷阱。即使它可以推断出某些输出的合理形状，它确定这些输出的规则是实现细节，为此需要付出性能代价或冒着跨 pandas 版本代码损坏的风险。如果知道函数将归约为标量，请始终选择 `pd.core.groupby.DataFrameGroupBy.agg` 代替 `pd.core.groupby.DataFrameGroupBy.apply`，后者仅用于极端用例。